# Principal Component Analysis via SVD — Solution

**Difficulty**: 🟢 Easy

---

### Problem Statement

PCA is the first tool people reach for when looking at high-dimensional data. It is used to
compress CLIP/DINO embeddings, to visualize learned features, to whiten inputs, to analyze
activations and gradients, and to initialize low-rank adapters. It finds the orthonormal directions
along which centered data varies most. Those are the top eigenvectors of the sample covariance

$$\Sigma = \frac{1}{N-1}\,X_c^{\top} X_c, \qquad X_c = X - \bar x .$$

The numerically sound way to get them is **not** to form $\Sigma$ at all. Take the **singular value
decomposition** of the centered data instead, $X_c = U\,\mathrm{diag}(s)\,V^{\top}$.

---

### Requirements

1. **`pca_fit(X, k) -> (components, explained_variance, mean)`**
   - `X`: `(N, D)` float tensor, one sample per row. `1 <= k <= min(N, D)`.
   - `mean`: `(D,)` per-feature mean.
   - `components`: `(k, D)` principal directions as **orthonormal rows**, sorted by decreasing variance.
   - `explained_variance`: `(k,)` variance of the data along each component, using the **unbiased
     $N-1$ convention**: $\lambda_i = s_i^2/(N-1)$. These equal the top-$k$ eigenvalues of
     `torch.cov(X.T)`.
   - Compute them from the SVD of the centered data.
2. **`pca_transform(X, components, mean)`** → `(N, k)`: coordinates $Z = (X - \bar x)\,C^{\top}$.
3. **`pca_inverse_transform(Z, components, mean)`** → `(N, D)`: reconstruction $\hat X = Z\,C + \bar x$.
4. The sign of each component is arbitrary ($v$ and $-v$ are both eigenvectors). Any sign is accepted.
5. Properties your implementation should satisfy (the tests check them):
   - projected coordinates are uncorrelated, with variance $\lambda_i$ along axis $i$;
   - $\frac{1}{N-1}\sum_n \lVert x_n - \hat x_n\rVert^2 = \sum_{i>k}\lambda_i$, the variance you discarded;
   - with `k = D` the reconstruction is exact, and `pca_transform(mean)` is $0$.

---

### Constraints

- ✅ `torch.linalg.svd` (use the thin SVD: `full_matrices=False`).
- ❌ Don't form the covariance / Gram matrix $X_c^{\top}X_c$ in `pca_fit` (see *Why it works* in the solution).
- ❌ No `torch.pca_lowrank`, `torch.svd_lowrank` or scikit-learn.
- ❌ Don't whiten (rescale) in `pca_transform`.

---

<details>
  <summary>💡 Hint</summary>

  - Why must you subtract the mean first? Where would the "direction of maximum variance" point if
    the data sat far from the origin?
  - Substitute $X_c = U S V^{\top}$ into $X_c^{\top} X_c$. What are its eigenvectors and eigenvalues?
  - Of $U$ (size $N \times r$) and $V$ (size $D \times r$), which one lives in feature space?
    `torch.linalg.svd` returns $V^{\top}$, already sorted by decreasing singular value.
  - $X_c V = U S$. What does that say about the covariance of the projected data?

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
# Anisotropic Gaussian: known standard deviations along a random rotation, plus an offset mean.
torch.manual_seed(0)
N, D = 500, 5
true_std = torch.tensor([4.0, 2.0, 1.0, 0.5, 0.1])
Q, _ = torch.linalg.qr(torch.randn(D, D))                      # random orthonormal basis
X = torch.randn(N, D) * true_std @ Q.T + torch.tensor([1.0, -2.0, 0.5, 3.0, 0.0])
print("X", tuple(X.shape), "| per-direction variances ~", [round(v, 3) for v in (true_std ** 2).tolist()])

In [ ]:
def pca_fit(X, k):
    """
    Args:
        X: (N, D) data, one sample per row.
        k: number of components, 1 <= k <= min(N, D).
    Returns:
        components:         (k, D) principal directions as orthonormal rows, by decreasing variance.
        explained_variance: (k,) variance of the data along each component (divides by N - 1).
        mean:               (D,) per-feature mean that was subtracted.
    """
    N, D = X.shape
    mean = X.mean(dim=0)                                        # (D,)
    # Thin SVD of the centered data: Xc = U diag(S) Vh. The rows of Vh are the
    # eigenvectors of Xc^T Xc, with eigenvalues S^2 — no covariance matrix needed.
    U, S, Vh = torch.linalg.svd(X - mean, full_matrices=False)  # S sorted descending
    components = Vh[:k]                                         # (k, D)
    explained_variance = S[:k] ** 2 / (N - 1)                   # unbiased covariance eigenvalues

    # Optional: fix the sign ambiguity (v and -v are both eigenvectors) so results are
    # reproducible — make the largest-magnitude entry of each component positive.
    pivot = components.abs().argmax(dim=1, keepdim=True)        # (k, 1)
    components = components * torch.sign(components.gather(1, pivot))
    return components, explained_variance, mean


def pca_transform(X, components, mean):
    """(N, D) -> (N, k) coordinates in the principal basis."""
    return (X - mean) @ components.T


def pca_inverse_transform(Z, components, mean):
    """(N, k) -> (N, D) reconstruction in the original space."""
    return Z @ components + mean

In [ ]:
# Test 1: explained variance = top-k eigenvalues of the sample covariance (oracle: eigh)
k = 3
components, explained_variance, mean = pca_fit(X, k)
assert components.shape == (k, D) and explained_variance.shape == (k,) and mean.shape == (D,)

evals, evecs = torch.linalg.eigh(torch.cov(X.T))      # ascending order; torch.cov divides by N - 1
evals, evecs = evals.flip(0), evecs.flip(1)           # largest first
print("explained_variance:", explained_variance)
print("top-k eigenvalues :", evals[:k])
assert torch.allclose(explained_variance, evals[:k], rtol=1e-3)
assert torch.allclose(components @ components.T, torch.eye(k), atol=1e-5), "rows must be orthonormal"
cos = (components * evecs[:, :k].T).sum(1).abs()     # equal up to sign
assert torch.allclose(cos, torch.ones(k), atol=1e-3), f"components != eigenvectors: |cos| = {cos}"
print("Test 1 passed")

In [ ]:
# Test 2: projection, reconstruction error, exact reconstruction at k = D
k = 3
components, explained_variance, mean = pca_fit(X, k)
evals = torch.linalg.eigvalsh(torch.cov(X.T)).flip(0)   # covariance eigenvalues, largest first
Z = pca_transform(X, components, mean)
assert Z.shape == (N, k)
assert torch.allclose(torch.cov(Z.T), torch.diag(explained_variance), atol=1e-3 * explained_variance[0].item()), \
    "projected coordinates should be uncorrelated with variance explained_variance"
assert pca_transform(mean[None], components, mean).abs().max() < 1e-5, "the mean must map to 0"

X_hat = pca_inverse_transform(Z, components, mean)
err = ((X - X_hat) ** 2).sum() / (N - 1)
print(f"reconstruction error {err:.5f} = discarded variance {evals[k:].sum():.5f}")
assert torch.isclose(err, evals[k:].sum(), rtol=1e-3)

C_all, _, m_all = pca_fit(X, D)
X_back = pca_inverse_transform(pca_transform(X, C_all, m_all), C_all, m_all)
assert torch.allclose(X_back, X, atol=1e-4), "k = D must reconstruct X exactly"
print("Test 2 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("pca", pca_fit=pca_fit, pca_transform=pca_transform, pca_inverse_transform=pca_inverse_transform)

In [ ]:
# Optional: why SVD of the data beats eigh of the covariance (float32, condition number 1e4)
torch.manual_seed(0)
n, d = 500, 4
Qd, _ = torch.linalg.qr(torch.randn(d, d, dtype=torch.float64))
Xd = (torch.randn(n, d, dtype=torch.float64) * torch.tensor([1.0, 1e-1, 1e-2, 1e-4], dtype=torch.float64)
      @ Qd.T + 10.0).float()                                  # float32 data, large offset
Xc = Xd - Xd.mean(0)
truth = torch.linalg.svdvals(Xc.double()) ** 2 / (n - 1)      # float64 reference
via_svd = torch.linalg.svdvals(Xc) ** 2 / (n - 1)
via_cov = torch.linalg.eigvalsh(Xc.T @ Xc / (n - 1)).flip(0)
print("float64 truth  :", [f"{v:.3e}" for v in truth.tolist()])
print("float32 SVD    :", [f"{v:.3e}" for v in via_svd.tolist()])
print("float32 cov+eig:", [f"{v:.3e}" for v in via_cov.tolist()])
print("relative error of the smallest eigenvalue: SVD "
      f"{abs(via_svd[-1].item() / truth[-1].item() - 1):.1e}, covariance {abs(via_cov[-1].item() / truth[-1].item() - 1):.1e}")

## Why it works

**PCA as an eigenproblem.** The variance of the data projected onto a unit vector $v$ is
$v^{\top}\Sigma v$. Maximizing this Rayleigh quotient gives the top eigenvector of $\Sigma$, and
maximizing it again in the orthogonal complement gives the next one (Courant–Fischer). The top-$k$
principal directions are therefore the top-$k$ eigenvectors, and the variance along each is the
eigenvalue.

**Getting it from the SVD.** With $X_c = U S V^{\top}$ (thin SVD, $U^{\top}U = V^{\top}V = I$):
$$X_c^{\top}X_c = V S U^{\top} U S V^{\top} = V S^2 V^{\top} \;\Rightarrow\; \Sigma = V\,\tfrac{S^2}{N-1}\,V^{\top}.$$
So the rows of $V^{\top}$ are the eigenvectors and $\lambda_i = s_i^2/(N-1)$, and `svd` already
sorts them in descending order. The projection is $Z = X_c V_k = U_k S_k$, whose columns are
orthogonal with variance $s_i^2/(N-1)$. That's why PCA coordinates are decorrelated.

**Reconstruction error.** $\hat X_c = X_c V_k V_k^{\top}$ is the best rank-$k$ approximation of $X_c$
in Frobenius norm (Eckart–Young), with $\lVert X_c - \hat X_c\rVert_F^2 = \sum_{i>k} s_i^2$. Dividing
by $N-1$ gives the sum of the discarded eigenvalues. With $k = D$, $V V^{\top} = I$ and the
reconstruction is exact.

**Why SVD and not `eigh(cov)`: conditioning.** Forming $X_c^{\top}X_c$ **squares the condition number**,
$\kappa(X_c^{\top}X_c) = \kappa(X_c)^2$. A backward-stable eigensolver on the covariance has
absolute error $\approx \varepsilon\,\lambda_{\max}$. In float32 ($\varepsilon \approx 6\cdot10^{-8}$),
any eigenvalue below about $10^{-7}\lambda_{\max}$, i.e. a singular value ratio of about $3\cdot10^{-4}$,
is pure rounding noise. It can even come out **negative**. The SVD works on $X_c$ directly, so its
error is relative to $s_{\max}$, not $s_{\max}^2$. The optional demo cell shows the smallest
eigenvalue: correct to about $10^{-5}$ via SVD, sign-flipped via the covariance.

**The $N-1$ convention** matches `torch.cov`, `np.cov` and scikit-learn's `explained_variance_`
(Bessel's correction: the sample mean used one degree of freedom). The directions don't depend
on it, only the reported variances.

**Sign.** Singular vectors are unique only up to sign. The reference makes the largest-magnitude
entry of each component positive, as scikit-learn's `svd_flip` does, so repeated runs and different
LAPACK builds give identical components.

## Complexity

- Thin SVD: $O(N D \min(N, D))$ time, $O(N D)$ memory. Transform / inverse: $O(N D k)$.
- When only $k \ll \min(N, D)$ components are needed: randomized SVD
  (`torch.svd_lowrank`, Halko et al.) costs $O(N D k)$.

## Common bugs

- **Not centering.** The first "component" then points at the mean, and every variance is off.
- **Dividing by $N$** instead of $N-1$. That's the MLE variance; it's 0.2% off at $N = 500$ and 2.5% at $N = 40$.
- **Returning $V$ (columns) instead of $V^{\top}$ (rows)**, or using $U$, which lives in sample space.
- **`torch.linalg.eigh` returns ascending eigenvalues.** Forgetting to flip gives the *least* important directions.
- **Centering new data with its own mean** in `transform` instead of the training mean.
- **Whitening by accident**: dividing $Z$ by $\sqrt{\lambda}$ or by $s$.
- **`full_matrices=True`** with $N \gg D$, which allocates an $N \times N$ $U$ you never use.

## Interview follow-ups

1. **When would you use `eigh` on the covariance anyway?** When $D \ll N$ and the data is streamed or
   sharded. You accumulate the $D \times D$ matrix $\sum x x^{\top}$ in float64 across batches or
   workers, then decompose it once. When $N \ll D$ (few images, huge feature vectors), use the
   $N \times N$ Gram matrix $X_c X_c^{\top}$ and map back with $V = X_c^{\top} U S^{-1}$. That's also
   the route to kernel PCA.
2. **What is whitening?** $Z_\text{white} = (X - \bar x)\,V_k\,\mathrm{diag}(1/\sqrt{\lambda_i + \epsilon})$
   gives identity covariance. ZCA whitening rotates back,
   $W = V\,\mathrm{diag}(1/\sqrt{\lambda+\epsilon})\,V^{\top}$, which stays close to the original space
   (classic image preprocessing). $\epsilon$ stops tiny-variance noise directions from being blown up.
3. **How do you choose $k$?** The explained-variance ratio $\lambda_i/\sum_j \lambda_j$ (e.g. keep 95%),
   a scree-plot elbow, or downstream validation. Probabilistic PCA gives a likelihood you can
   cross-validate.
4. **PCA on huge data?** Randomized SVD (a few power iterations on a random sketch), incremental /
   streaming PCA (e.g. Oja's rule), or power iteration with deflation for the top few components.
5. **Should you standardize features first?** If features have different units or scales, yes.
   Otherwise the largest-scale feature dominates the first component. That amounts to PCA on the
   correlation matrix.
6. **How does PCA relate to a linear autoencoder?** A linear autoencoder trained with MSE learns
   the same $k$-dimensional subspace as PCA. Its weights span it but needn't be orthonormal or
   ordered. Nonlinear autoencoders generalize PCA to curved manifolds.